# Chapter 1: The Agentic AI Testing Problem

**Monograph:** *How to Test and Validate Agentic AI Systems*

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Articulate why testing agentic AI is fundamentally harder than testing classifiers or RAG pipelines.
2. Identify the six failure modes unique to tool-using agents.
3. Observe the limitations of string-match scoring on real agent output.
4. Understand the GMSH testing philosophy and what the rest of the monograph will cover.

## Prerequisites

- No API keys required. All exercises use a deterministic **MockHarness** as the system under test.
- The test document `model_risk_assessment.md` is bundled with the GMSH package.

---
## Preamble

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# [AUTO-INJECTED] Disable DOE tracing for parallel execution safety
from knowlytix.harness.doe import DOEHarnessConfig
DOEHarnessConfig.__dataclass_fields__['enable_tracing'].default = False


In [ ]:
from importlib.resources import files
# Load the running example: a model risk assessment report
MD_PATH = str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md"))

with open(MD_PATH) as f:
    context = f.read()

print(f"Document loaded: {len(context):,} characters")
print(f"First 300 chars:\n{context[:300]}")

---
## 1.1 The Running Example

Throughout this monograph we use a single source document: the **Meridian Financial Technologies Model Risk Assessment Report**. It is a realistic enterprise governance document containing:

- 12 models under review (MDL-001 through MDL-012)
- Accuracy metrics (AUC, Gini, KS statistic, Brier score)
- Robustness ratings under noise perturbation
- Fairness metrics for credit and lending models
- Data quality assessments (PSI drift scores, missing data rates)
- Risk findings with severity and remediation deadlines
- Model dependency chains and business impact quantification

Imagine deploying an AI agent to assist model validators at Meridian. The agent should be able to look up specific metrics, identify compliance violations, trace dependency chains and escalate critical findings. But how do we know whether the agent is doing these things *correctly*?

---
## 1.2 Meet the System Under Test: MockHarness

Our system under test (SUT) is a deterministic **MockHarness** that simulates a tool-using agentic AI. It:

1. Parses the question to decide which tools to call.
2. Executes tools (`data_lookup`, `risk_check`, `calculate`, `submit_escalation`, `recall_memory`).
3. Composes an answer from the tool results.
4. Handles blocked tools via a configurable recovery strategy.

This is *not* an LLM. It is a rule-based mock that makes realistic mistakes -- exactly the kind of mistakes a real agent would make. That makes it ideal for learning how to test agents: the bugs are there by design.

In [ ]:
from knowlytix.harness.fixtures import MockHarness, create_mock_evaluator, MOCK_TOOLS

# Inspect the tools available to the agent
print("Tools available to MockHarness:")
print("-" * 50)
for name, spec in MOCK_TOOLS.items():
    print(f"  {name:25s} [{spec['category']}]")
    print(f"    {spec['description']}")

In [6]:
# Create an evaluator: a callable (question, context) -> answer
evaluator = create_mock_evaluator(recovery_mode="retry")

# Ask a simple question
answer = evaluator("What is the status of MDL-003?", context)
print(f"Answer: {answer}")

Answer: Degraded


The agent got this one right. But will it always be this easy? Let's find out.

---
## Lab 1.1 -- See the Problem Firsthand

We will run the mock agent on a battery of questions drawn from the model risk assessment. For each question we know the correct answer from the document. We score with **exact string matching** -- the simplest possible evaluation method.

Pay attention to *why* each failure occurs. The failures fall into distinct categories that will motivate the testing framework we build in later chapters.

In [7]:
# Define a battery of questions with known ground truth
questions = [
    {
        "question": "What is the test AUC of MDL-001?",
        "ground_truth": "0.8412",
        "category": "exact_recall",
        "note": "Numeric lookup from Table 3.1",
    },
    {
        "question": "What domain is MDL-002 in?",
        "ground_truth": "Fraud Detection",
        "category": "exact_recall",
        "note": "String lookup from Table 2.1",
    },
    {
        "question": "How many models are rated as High risk?",
        "ground_truth": "4",
        "category": "counting",
        "note": "Count from Executive Summary or Table 2.1",
    },
    {
        "question": "Is model MDL-001 status Degraded?",
        "ground_truth": "True",
        "category": "boolean",
        "note": "Status check from Table 2.1",
    },
    {
        "question": "What is the risk rating of MDL-004?",
        "ground_truth": "High",
        "category": "exact_recall",
        "note": "Direct lookup from Table 2.1",
    },
    {
        "question": "Does MDL-005 have test AUC above 0.75?",
        "ground_truth": "False",
        "category": "threshold",
        "note": "MDL-005 test AUC is 0.7234, which is below 0.75",
    },
    {
        "question": "What is the composite risk score of MDL-001?",
        "ground_truth": "56.75",
        "category": "cross_reference",
        "note": "From Table 5.2 -- requires looking beyond basic metrics",
    },
    {
        "question": "If MDL-009 fails, what downstream model is affected in credit scoring?",
        "ground_truth": "MDL-001",
        "category": "multi_hop",
        "note": "From Table 6.1: MDL-009 -> MDL-001 (PD feeds into credit scoring)",
    },
]

In [8]:
# Run the evaluator and score with exact string matching
def exact_match(answer, ground_truth):
    """Case-insensitive exact string match."""
    return str(answer).strip().lower() == str(ground_truth).strip().lower()

print(f"{'#':<3} {'Category':<16} {'Match':<6} {'Answer':<30} {'Ground Truth':<20}")
print("=" * 80)

n_correct = 0
results = []
for i, q in enumerate(questions, 1):
    answer = evaluator(q["question"], context)
    match = exact_match(answer, q["ground_truth"])
    n_correct += match
    results.append({**q, "answer": answer, "match": match})

    answer_display = str(answer)[:28]
    symbol = "PASS" if match else "FAIL"
    print(f"{i:<3} {q['category']:<16} {symbol:<6} {answer_display:<30} {q['ground_truth']:<20}")

print("=" * 80)
print(f"Score: {n_correct}/{len(questions)} ({100*n_correct/len(questions):.1f}%)")

#   Category         Match  Answer                         Ground Truth        
1   exact_recall     FAIL   0.8934                         0.8412              
2   exact_recall     PASS   Fraud Detection                Fraud Detection     
3   counting         FAIL   High                           4                   
4   boolean          PASS   True                           True                
5   exact_recall     PASS   High                           High                
6   threshold        FAIL   0.7891                         False               
7   cross_reference  FAIL   0.8934                         56.75               
8   multi_hop        FAIL   | MDL-009 | PD-Logistic | Pr   MDL-001             
Score: 3/8 (37.5%)


### What happened?

The mock agent scores about **3/8 (37.5%)** with exact string matching. Let's examine the failures one by one.

In [9]:
# Examine each failure in detail
print("DETAILED FAILURE ANALYSIS")
print("=" * 70)

for r in results:
    if not r["match"]:
        print(f"\nQuestion:     {r['question']}")
        print(f"Ground truth: {r['ground_truth']}")
        print(f"Agent answer: {str(r['answer'])[:120]}")
        print(f"Category:     {r['category']}")
        print(f"Note:         {r['note']}")
        print("-" * 70)

DETAILED FAILURE ANALYSIS

Question:     What is the test AUC of MDL-001?
Ground truth: 0.8412
Agent answer: 0.8934
Category:     exact_recall
Note:         Numeric lookup from Table 3.1
----------------------------------------------------------------------

Question:     How many models are rated as High risk?
Ground truth: 4
Agent answer: High
Category:     counting
Note:         Count from Executive Summary or Table 2.1
----------------------------------------------------------------------

Question:     Does MDL-005 have test AUC above 0.75?
Ground truth: False
Agent answer: 0.7891
Category:     threshold
Note:         MDL-005 test AUC is 0.7234, which is below 0.75
----------------------------------------------------------------------

Question:     What is the composite risk score of MDL-001?
Ground truth: 56.75
Agent answer: 0.8934
Category:     cross_reference
Note:         From Table 5.2 -- requires looking beyond basic metrics
-------------------------------------------------

### Observations

Each failure reveals a different testing challenge:

| Question | What Went Wrong | Testing Challenge |
|----------|----------------|-------------------|
| Test AUC of MDL-001 | Agent returned **train AUC** (0.8934) instead of **test AUC** (0.8412) | The answer is a valid AUC for MDL-001 -- just the *wrong* AUC. String matching cannot distinguish which metric was requested. |
| How many High risk? | Agent returned **"High"** instead of the count **4** | Agent misunderstood the question type entirely -- it answered *what* risk level instead of *how many*. |
| AUC above 0.75? | Agent returned **0.7891** instead of **False** | Agent returned the numeric value instead of evaluating the threshold comparison. The value it returned (train AUC) also differs from the test AUC. |
| Composite risk score? | Agent returned **0.8934** instead of **56.75** | Agent looked up the wrong table entirely -- it found an AUC metric instead of the composite risk score from a different section. |
| Downstream of MDL-009? | Agent returned a **table row** instead of **MDL-001** | Agent could not perform multi-hop reasoning: "MDL-009 fails" -> Table 6.1 dependency chain -> downstream model. |

These are not random errors. They are **systematic failure modes** that any tool-using agent can exhibit.

---
## Lab 1.2 -- Explore the Mock Agent's Tool Behavior

An agentic AI system does not just produce text -- it *acts*. It chooses tools, sequences them and reacts to failures. Testing must therefore go beyond the final answer and examine the **process** by which the answer was produced.

Let's look at the tool call traces.

In [10]:
# Create a MockHarness so we can inspect tool traces directly
harness = MockHarness(recovery_mode="retry")

prompt = (
    f"<report>\n{context}\n</report>\n\n"
    f"Answer the following question using the report and any tools "
    f"available to you.\n\nWhat is the test AUC of MDL-001?"
)

answer = harness.run(prompt)

print(f"Answer: {answer}")
print(f"\nTool calls ({len(harness._tool_records)} total):")
for i, rec in enumerate(harness._tool_records, 1):
    print(f"  [{i}] {rec['tool']}")
    print(f"      Args:    {rec['args']}")
    print(f"      Blocked: {rec['blocked']}")
    print(f"      Result:  {str(rec['result'])[:100]}")

Answer: 0.8934

Tool calls (2 total):
  [1] data_lookup
      Args:    {'entity': 'MDL-001'}
      Blocked: False
      Result:  {'name': 'CreditScore-XGB', 'domain': 'Credit Decisioning', 'risk': 'High', 'status': 'Degraded', 'a
  [2] risk_check
      Args:    {'model_id': 'MDL-001'}
      Blocked: False
      Result:  Model MDL-001 (CreditScore-XGB): Risk=High, Status=Degraded, Domain=Credit Decisioning


The agent called `data_lookup` and `risk_check` for MDL-001. The `data_lookup` tool returned the full model data dictionary -- which contains *both* `train_auc: 0.8934` and `test_auc: 0.8412`. The agent then picked the first numeric value it encountered (`0.8934`) rather than the specific metric requested.

The tool returned the right data; the agent extracted the wrong value. **The process was partially correct but the outcome was wrong.** This is exactly the kind of nuance that testing must capture.

### Recovery Behavior Under Tool Failure

What happens when a tool the agent depends on becomes unavailable? The MockHarness supports three recovery strategies:

- **retry**: note the failure and continue with remaining tool results
- **fallback**: try `recall_memory` as a backup information source
- **escalate**: call `submit_escalation` to flag the blocked tool

The same question can produce radically different tool traces depending on the recovery mode.

In [11]:
# Block the risk_check tool and compare recovery behaviors
def block_risk_check(name, args):
    """Hook that blocks risk_check but allows everything else."""
    return name != "risk_check"

question = "What is the risk rating of MDL-001?"

print(f"Question: {question}")
print("Condition: risk_check tool is BLOCKED")
print("=" * 70)

for mode in ["retry", "fallback", "escalate"]:
    h = MockHarness(recovery_mode=mode)
    h.hooks.before_tool = block_risk_check

    prompt = (
        f"<report>\n{context}\n</report>\n\n"
        f"Answer the following question using the report and any tools "
        f"available to you.\n\n{question}"
    )
    answer = h.run(prompt)

    tools_used = [rec["tool"] for rec in h._tool_records]
    blocked = [rec["tool"] for rec in h._tool_records if rec["blocked"]]

    print(f"\n--- {mode.upper()} mode ---")
    print(f"  Tools called:   {tools_used}")
    print(f"  Tools blocked:  {blocked}")
    print(f"  Answer:         {str(answer)[:100]}")
    print(f"  Correct (High): {'high' in answer.lower() if isinstance(answer, str) else False}")

Question: What is the risk rating of MDL-001?
Condition: risk_check tool is BLOCKED

--- RETRY mode ---
  Tools called:   ['data_lookup', 'risk_check']
  Tools blocked:  ['risk_check']
  Answer:         {'name': 'CreditScore-XGB', 'domain': 'Credit Decisioning', 'risk': 'High', 'status': 'Degraded', 'a
  Correct (High): True

--- FALLBACK mode ---
  Tools called:   ['data_lookup', 'risk_check', 'recall_memory']
  Tools blocked:  ['risk_check']
  Answer:         Tool 'risk_check' blocked; falling back to memory. # Model Risk Assessment Report — Enterprise AI Go
  Correct (High): True

--- ESCALATE mode ---
  Tools called:   ['data_lookup', 'risk_check', 'submit_escalation']
  Tools blocked:  ['risk_check']
  Answer:         {'name': 'CreditScore-XGB', 'domain': 'Credit Decisioning', 'risk': 'High', 'status': 'Degraded', 'a
  Correct (High): True


### Observations

The three recovery modes produce different tool sequences and different answers:

| Mode | Tool Sequence | Answer Quality |
|------|--------------|----------------|
| **retry** | `data_lookup` -> `risk_check` (blocked) | Returns raw data dict; contains "High" but in an unstructured dump |
| **fallback** | `data_lookup` -> `risk_check` (blocked) -> `recall_memory` | Falls back to searching the document text |
| **escalate** | `data_lookup` -> `risk_check` (blocked) -> `submit_escalation` | Escalates the failure instead of answering |

This illustrates a fundamental challenge: **testing an agent requires evaluating not just the answer but also the tool-use process.** Did the agent use the right tools? Did it recover appropriately? Did it escalate when it should have?

---
## Lab 1.3 -- The Six Failure Modes of Agentic AI

Based on what we have observed, we can identify six distinct failure modes that are unique to tool-using agents. Each one requires a different testing strategy.

### Failure Mode 1: Factual Hallucination

The agent asserts something that is not in the source document or contradicts it.

In [12]:
# Factual hallucination: agent returns train AUC instead of test AUC
answer = evaluator("What is the test AUC of MDL-001?", context)
print("Question:     What is the test AUC of MDL-001?")
print("Ground truth: 0.8412 (from Table 3.1)")
print(f"Agent answer: {answer}")
print("\nThe agent returned a real value from the document (train AUC = 0.8934),")
print("but it is the WRONG metric. This is a subtle hallucination -- the agent")
print("did not fabricate a number, but it attributed the wrong value to the query.")

Question:     What is the test AUC of MDL-001?
Ground truth: 0.8412 (from Table 3.1)
Agent answer: 0.8934

The agent returned a real value from the document (train AUC = 0.8934),
but it is the WRONG metric. This is a subtle hallucination -- the agent
did not fabricate a number, but it attributed the wrong value to the query.


### Failure Mode 2: Tool Misuse

The agent calls the wrong tool, calls tools in the wrong order or passes incorrect arguments.

In [13]:
# Tool misuse: escalation question triggers data_lookup instead
harness = MockHarness(recovery_mode="retry")
prompt = (
    f"<report>\n{context}\n</report>\n\n"
    f"Answer the following question using the report and any tools "
    f"available to you.\n\n"
    f"Escalate the MDL-001 fairness finding for human review."
)
answer = harness.run(prompt)

print("Question:      Escalate the MDL-001 fairness finding for human review.")
print("Expected tool: submit_escalation")
print(f"Actual tools:  {[r['tool'] for r in harness._tool_records]}")
print("\nThe agent should have called submit_escalation, but instead it called")
print("data_lookup and risk_check -- tools for retrieving data, not for acting.")
print("The agent used the wrong tools for the task.")

Question:      Escalate the MDL-001 fairness finding for human review.
Expected tool: submit_escalation
Actual tools:  ['data_lookup', 'risk_check']

The agent should have called submit_escalation, but instead it called
data_lookup and risk_check -- tools for retrieving data, not for acting.
The agent used the wrong tools for the task.


### Failure Mode 3: Policy Violation

The agent performs an action that should require authorization or is explicitly forbidden.

In [14]:
# Policy violation: imagine a rule that submit_escalation requires human approval
# (block_risk_check was defined in Lab 1.2 above; redefine here for safety)
def block_risk_check(name, args):
    return name != "risk_check"

print("Scenario: A policy states that escalations require human approval.")
print("The agent calls submit_escalation directly without seeking approval.")
print()

harness = MockHarness(recovery_mode="escalate")
# Simulate blocked risk_check to trigger escalation behavior
harness.hooks.before_tool = block_risk_check

prompt = (
    f"<report>\n{context}\n</report>\n\n"
    f"Answer the following question using the report and any tools "
    f"available to you.\n\n"
    f"What is the risk rating of MDL-001?"
)
answer = harness.run(prompt)

print(f"Tools called: {[r['tool'] for r in harness._tool_records]}")
print("\nThe 'escalate' recovery mode called submit_escalation automatically")
print("when risk_check was blocked. In a production system with policy rules,")
print("this automatic escalation might violate a 'requires human_approval' policy.")
print("\nThe agent's recovery strategy inadvertently violated a governance rule.")

Scenario: A policy states that escalations require human approval.
The agent calls submit_escalation directly without seeking approval.

Tools called: ['data_lookup', 'risk_check', 'submit_escalation']

The 'escalate' recovery mode called submit_escalation automatically
when risk_check was blocked. In a production system with policy rules,
this automatic escalation might violate a 'requires human_approval' policy.

The agent's recovery strategy inadvertently violated a governance rule.


### Failure Mode 4: Inconsistent Multi-Hop Reasoning

The agent fails to chain information across multiple tables or sections.

In [15]:
# Multi-hop failure: agent cannot trace the dependency chain
answer = evaluator(
    "If MDL-009 fails, what downstream model is affected in credit scoring?",
    context,
)
print("Question:     If MDL-009 fails, what downstream model is affected")
print("              in credit scoring?")
print("Ground truth: MDL-001")
print("              (Table 6.1: MDL-009 -> MDL-001, PD feeds credit scoring)")
print(f"Agent answer: {str(answer)[:100]}")
print("\nThe agent needed to: (1) find MDL-009 in Table 6.1, (2) identify its")
print("downstream models, (3) filter for 'credit scoring'. This multi-hop")
print("reasoning chain is beyond what the mock agent can do.")

Question:     If MDL-009 fails, what downstream model is affected
              in credit scoring?
Ground truth: MDL-001
              (Table 6.1: MDL-009 -> MDL-001, PD feeds credit scoring)
Agent answer: | MDL-009 | PD-Logistic | Probability of Default | 2 | Medium | Stable |

The agent needed to: (1) find MDL-009 in Table 6.1, (2) identify its
downstream models, (3) filter for 'credit scoring'. This multi-hop
reasoning chain is beyond what the mock agent can do.


### Failure Mode 5: Confidence Miscalibration

The agent expresses confidence that does not match its actual accuracy.

In [ ]:
# Confidence miscalibration: use FullMockHarness with confidence mode
from knowlytix.harness.fixtures import create_full_evaluator

confident_evaluator = create_full_evaluator(confidence_mode=True)

# The agent will append [Confidence: 8/10] to every answer -- even wrong ones
test_cases = [
    ("What is the test AUC of MDL-001?", "0.8412"),
    ("What domain is MDL-002 in?", "Fraud Detection"),
    ("How many models are rated as High risk?", "4"),
]

print(f"{'Question':<50} {'Correct?':<9} {'Agent Output'}")
print("=" * 100)
for q, gt in test_cases:
    answer = confident_evaluator(q, context)
    is_correct = gt.lower() in answer.lower()
    print(f"{q:<50} {'YES' if is_correct else 'NO':<9} {answer[:50]}")

print("\nThe agent expresses high confidence (8/10) on every answer, including")
print("the ones it gets wrong. A well-calibrated agent should express lower")
print("confidence when it is more likely to be wrong.")

### Failure Mode 6: State Corruption Across Turns

In multi-turn interactions the agent's internal state can degrade, causing it to forget or confuse information.

In [ ]:
# State corruption: teach a fact, distract, then test recall
from knowlytix.harness.fixtures import FullMockHarness

harness = FullMockHarness(recovery_mode="retry")

# Teach a fact
harness.teach("project_deadline", "2026-06-15")
print("Taught: project_deadline = 2026-06-15")

# Distractor turns (these interact with the model risk context)
prompt_template = (
    "<report>\n" + context[:500] + "\n</report>\n\n"
    "Answer the following question using the report and any tools "
    "available to you.\n\n{}"
)

_ = harness.run(prompt_template.format("What is the test AUC of MDL-001?"))
_ = harness.run(prompt_template.format("What domain is MDL-002 in?"))
_ = harness.run(prompt_template.format("What is the risk rating of MDL-004?"))
print("Completed 3 distractor turns.")

# Now test recall
recall = harness.run("When is the project deadline?")
print(f"\nRecall after 3 turns: {recall}")
print("Expected:             2026-06-15")
print(f"Match:                {recall.strip() == '2026-06-15'}")
print("\nThe mock's recall failed because its keyword matching cannot connect")
print("the query 'project deadline' (space) to the stored key 'project_deadline'")
print("(underscore). This is a deterministic failure of the mock's simple logic.")
print("In a real LLM agent, the failure mode is different: facts taught via")
print("conversation context degrade as the context window fills and older")
print("turns are compressed or dropped.")


---
## Lab 1.4 -- Why String Matching Fails

String matching is the most common evaluation method: compare the agent's output to the expected answer character by character. It is fast, deterministic and easy to implement. But it has two fatal flaws:

1. **False negatives:** The answer is semantically correct but does not match the expected string.
2. **False positives:** The answer matches the expected string but is wrong in context.

Let's see both.

In [18]:
# FALSE NEGATIVES: correct answers that fail string matching
print("FALSE NEGATIVES -- correct answers that fail exact match")
print("=" * 60)

false_negatives = [
    ("0.8412",       "0.84",          "Rounded but correct to 2 decimal places"),
    ("High",         "Risk=High",      "Correct value embedded in structured output"),
    ("High",         "high",           "Correct but different capitalization"),
    ("Fraud Detection", "fraud detection", "Correct but lowercased"),
    ("4",            "four",           "Correct but spelled out"),
    ("4",            "There are 4 high-risk models", "Correct but in a sentence"),
]

for gt, answer, reason in false_negatives:
    match = gt.strip() == answer.strip()
    print(f"  GT: {gt!r:20s}  Answer: {answer!r:35s}  Match: {match}")
    print(f"    -> {reason}")

FALSE NEGATIVES -- correct answers that fail exact match
  GT: '0.8412'              Answer: '0.84'                               Match: False
    -> Rounded but correct to 2 decimal places
  GT: 'High'                Answer: 'Risk=High'                          Match: False
    -> Correct value embedded in structured output
  GT: 'High'                Answer: 'high'                               Match: False
    -> Correct but different capitalization
  GT: 'Fraud Detection'     Answer: 'fraud detection'                    Match: False
    -> Correct but lowercased
  GT: '4'                   Answer: 'four'                               Match: False
    -> Correct but spelled out
  GT: '4'                   Answer: 'There are 4 high-risk models'       Match: False
    -> Correct but in a sentence


In [19]:
# FALSE POSITIVES: wrong answers that would pass a loose match
print("\nFALSE POSITIVES -- wrong answers that could pass loose matching")
print("=" * 60)

false_positives = [
    ("0.8412", "0.8934",
     "Both are AUC values for MDL-001 but from different columns (test vs train)"),
    ("4", "4",
     "'How many High risk?' vs 'How many models are Degraded?' -- same number, different fact"),
    ("MDL-001", "MDL-001",
     "Right entity name but agent found it for the wrong reason (guessed first model)"),
]

for gt, answer, reason in false_positives:
    match = gt.strip() == answer.strip()
    print(f"  GT: {gt!r:20s}  Answer: {answer!r:20s}  Match: {match}")
    print(f"    -> {reason}")


FALSE POSITIVES -- wrong answers that could pass loose matching
  GT: '0.8412'              Answer: '0.8934'              Match: False
    -> Both are AUC values for MDL-001 but from different columns (test vs train)
  GT: '4'                   Answer: '4'                   Match: True
    -> 'How many High risk?' vs 'How many models are Degraded?' -- same number, different fact
  GT: 'MDL-001'             Answer: 'MDL-001'             Match: True
    -> Right entity name but agent found it for the wrong reason (guessed first model)


### The Evaluation Trilemma

We face a three-way tradeoff:

| Method | Precision | Recall | Cost |
|--------|-----------|--------|------|
| **Exact string match** | High (few false positives) | Low (many false negatives) | Free |
| **LLM-as-judge** | Medium | Medium | Expensive; circular (using an LLM to evaluate an LLM) |
| **Human review** | High | High | Very expensive; not reproducible |

What we need is a fourth option: an evaluation method that is **precise** (few false positives), **comprehensive** (few false negatives), **cheap** (no LLM calls or human effort) and **auditable** (every decision can be traced to a verifiable fact).

That is what Geometric Memory Systems provide. By embedding the source document into a trained geometric space, we can evaluate agent answers using continuous geometric signals -- geodesic distance, tension energy, holonomy defects -- that go far beyond binary string matching.

The rest of this monograph shows you how.

---
## Lab 1.5 -- Preview: What GMSH Testing Looks Like

Before we close this chapter, here is a quick preview of what a complete GMSH evaluation looks like. We will cover every step in detail in later chapters. For now, just observe the *shape* of the output.

In [ ]:
from importlib.resources import files
from knowlytix.harness.doe import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="monograph/lab_stores",
    store_name="ch01_preview",
    factor_group="quick_screen",
    n_runs=4,
    method="sobol",
    seed=42,
    max_per_category=3,
    ingest_epochs=200,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)

# Step 1: Ingest document into a GMS expert store (ground truth)
print("[Step 1] Ingesting document into GMS expert store...")
store = bench.ingest()
stats = store.stats()
print(f"  Entities:    {stats['entities']}")
print(f"  Relations:   {stats['relations']}")
print(f"  Triples:     {stats['triples']}")
print(f"  ENM entries: {stats['enm_entries']}")

In [21]:
# Step 2: Generate DOE-expanded test questions from the geometric structure
print("[Step 2] Generating test questions...")
doe_qs = bench.generate_questions()

from collections import Counter
cats = Counter(dq.question.category for dq in doe_qs)
print(f"  Total DOE questions: {len(doe_qs)}")
print("  Categories:")
for cat, count in sorted(cats.items()):
    print(f"    {cat:<25s} {count:>3} questions")

[Step 2] Generating test questions...
  Base questions: 48 (20 generators, max 3/cat)

  DOE expansion: 48 base x 4 runs = 192 questions
  Factors: ['clarity', 'entity_aliasing', 'reasoning_cue']
  [BackgroundTrain] Starting: 5 entities, 0 relations, 25 triples
  [BackgroundTrain] Done: Trained: 5 entities, 0 relations, 50 steps in 3.9s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  Total DOE questions: 192
  Categories:
    adversarial                12 questions
    boundary                   12 questions
    counting                   12 questions
    cross_reference            12 questions
    exact_retrieval            12 questions
    governance                 12 questions
    holonomy                   12 questions
    link_prediction            12 questions
    multi_hop                  12 questions
    multi_hop_transport        12 questions
    phase_comparison           12 questions
    plausibility               12 questions
    recovery

In [ ]:
# Step 3: Validate ground truth (GMS answers its own questions)
print("[Step 3] Running GMS baseline (ground truth validation)...")
baseline = bench.run_gms_baseline()
print(f"  GMS baseline accuracy: {baseline.accuracy:.0%}")
print(f"  ({baseline.total_correct}/{baseline.total_questions} correct)")
print("\n  This confirms the ground truth is self-consistent.")

[Step 3] Running GMS baseline (ground truth validation)...
  [BackgroundTrain] Starting: 4 entities, 0 relations, 10 triples
  [BackgroundTrain] Done: Trained: 4 entities, 0 relations, 50 steps in 3.5s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved

  GMS baseline: 192 questions...
  [BackgroundTrain] Starting: 2 entities, 0 relations, 11 triples
  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 3.4s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 1 entities, 0 relations, 11 triples
  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 3.3s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 1 entities, 0 relations, 11 triples
  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 3.3s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTr

In [ ]:
# Step 4: Evaluate the mock agent against the same questions
print("[Step 4] Evaluating MockHarness against DOE questions...")
mock_evaluator = create_mock_evaluator(recovery_mode="fallback")
result = bench.run(evaluator=mock_evaluator)
print(f"  Mock agent accuracy: {result.accuracy:.1%}")
print(f"  ({result.total_correct}/{result.total_questions} correct)")

print("\n  Per-category breakdown:")
df = result.to_dataframe()
cat_acc = df.groupby("category")["correct"].agg(["sum", "count"])
for cat, row in cat_acc.iterrows():
    acc = row["sum"] / row["count"] if row["count"] > 0 else 0
    print(f"    {cat:<25s} {int(row['sum']):>2}/{int(row['count']):<3} ({acc:.0%})")

In four steps, GMSH has:

1. **Built a geometric ground truth** from the source document -- not just key-value pairs but a trained knowledge graph on the hypersphere with Cayley rotor relations, tension energy semantics and exact numerical memory.

2. **Generated questions automatically** from the geometric structure -- questions that exercise specific GMS primitives (plausibility scoring, contradiction detection, path consistency, link prediction, multi-hop transport, numeric comparison).

3. **Validated the ground truth** by confirming the GMS can answer its own questions with ~100% accuracy.

4. **Evaluated the SUT** with per-category accuracy and geometric confidence scores.

The subsequent chapters will teach you every component of this pipeline in depth.

---
## Exercises

### Exercise 1.1

Write three additional questions about the model risk assessment report (one `exact_recall`, one `threshold`, one `multi_hop`). Run the mock agent on them and classify the failure mode for any incorrect answers.

### Exercise 1.2

Modify the `block_risk_check` hook from Lab 1.2 to block `data_lookup` instead. How does the mock agent behave under each recovery mode when its primary information source is unavailable? Which mode produces the best answer?

### Exercise 1.3

Write a `fuzzy_match` scorer that handles the false-negative cases from Lab 1.4 (rounding, capitalization, embedded values). Test it on the 8 questions from Lab 1.1. Does it improve the score? Does it introduce any false positives?

In [23]:
# === Exercise 1.1: Your code here ===



In [24]:
# === Exercise 1.2: Your code here ===



In [25]:
# === Exercise 1.3: Your code here ===



---
## Summary

In this chapter we established the core problem:

1. **Agentic AI systems are harder to test** than classifiers because they make autonomous decisions about tool use, sequencing and recovery.

2. **Six failure modes** are unique to tool-using agents: factual hallucination, tool misuse, policy violation, inconsistent multi-hop reasoning, confidence miscalibration and state corruption.

3. **String matching is insufficient** as an evaluation method -- it produces both false negatives (correct answers rejected) and false positives (wrong answers accepted).

4. **We need geometric verification**: a method that is precise, comprehensive, cheap and auditable. GMSH provides this by embedding source documents into a trained geometric space and evaluating agent answers against six geometric primitives.

In the next chapter we will learn how Geometric Memory Systems work -- the mathematical substrate that makes this kind of testing possible.